In [ ]:
import pandas as pd
import numpy as np
import random
import string
import itertools


generate tables:
1. Product table: sku, product_name, cost (how much money it costs to make the product), price (how much we charge customers), brand ()
2. Orders: order_id, sku, quantity, date, product_price (how much a customer paid for these products, maybe not needed here? maybe price per unit)
    - maybe date should be in one format but in bigquery we transform to another one?
3. Inventory table: sku, current_stock (product units), expiration_date (expiration of the current stock)
    - assumption: in real world example, this table would be updated regularly. for simplicity for this project, our table is static and a "snapshot" for this particular date.

sku is the only unique variable here. product name MIGHT be the same

In [142]:
# PRODUCT TABLE

# sku: random 150 unique 3 letter string
# product_name: create a list of beauty items (e.g., conditioner, shampoo, etc)
# category: create a 
# brand: create a list of 7 fake brands
# demand: low, medium, high
# price: random(5,25)
# cost: random(0.4-0.7) * price

product_name =[
    ('Shampoo','Haircare'),
    ('Conditioner','Haircare'),
    ('Hair Mask','Haircare'),
    ('Hair Oil','Haircare'),
    ('Hair Serum','Haircare'),
    ('Bond Builder','Haircare'),
    ('Scalp Exfoliator','Haircare'),
    ('Dry Shampoo','Haircare'),
    ('Heat Protectant','Haircare'),
    ('Hairspray','Haircare'),
    ('Gel','Haircare'),
    ('Sea Salt Spray','Haircare'),
    ('Mousse','Haircare'),
    ('Cleanser','Skincare'),
    ('Toner','Skincare'),
    ('Serum','Skincare'),
    ('Eye Cream','Skincare'),
    ('Moisturiser','Skincare'),
    ('Sunscreen','Skincare'),
    ('Face Mask','Skincare'),
    ('Lip Balm','Skincare'),
    ('Spot Treatment','Skincare'),
    ('Exfoliator','Skincare'),
    ('Shower Gel','Bodycare'),
    ('Shower Oil','Bodycare'),
    ('Bar Soap','Bodycare'),
    ('Body Scrub','Bodycare'),
    ('Lotion','Bodycare'),
    ('Body Butter','Bodycare'),
    ('Body Oil','Bodycare'),
    ('Foot Cream','Bodycare'),
    ('Deodorant','Bodycare'),
    ('Razor','Bodycare'),
    ('Shaving Cream','Bodycare'),
    ('Loofah','Bodycare'),
    ('Bath Bomb','Bodycare'),
    ('Bath Salt','Bodycare'),
    ('Body Serum','Bodycare'),
    ('Acetone','Nailcare'),
    ('Nail Strengthener','Nailcare'),
    ('Cuticle Oil','Nailcare'),
    ('Nail File','Nailcare'),
    ('Nail Clippers','Nailcare'),
    ('Cuticle Pusher','Nailcare'),
    ('Cuticle Remover','Nailcare'),
    ('Nail Polish','Nailcare')
]

brand = ['LuxBeauty','EurBeauty','CareForSelf','LovelyBeauty','GlowAndShow','RenewBeauty','Skinore','PamperBeauty']

demand = ['High', 'Medium', 'Low']

cost_fracts = [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]

# generate a unique list of 150 skus. we take the alphabet, generate all unique combinations of 3 letters, return 15 random ones
            # generated_sku_nums = np.random.choice(range(100000,199999), size=150, replace=False) # replace makes sure we only get unique numbers
            # sku_list = ['ORD' + str(num) for num in generated_sku_nums] THIS IS ORDER LIST!!!

alph = string.ascii_uppercase
all_sku_combs = [''.join(let) for let in itertools.product(alph, repeat=3)]
sku_list = random.sample(all_sku_combs, k=150)

# for each unique sku, we select a random product+category pair and assign other values. this means that a product+category pair can be assigned to several different skus
generated_products = []
i = 0
for sku in sku_list: 
    price = random.choice(range(5,26))
    cost = price * random.choice(cost_fracts)
    prod,categ = random.choice(product_name)
    product = {
        'sku': sku,
        # product_name = [('a','b'),('a','b'),('a','b')]
        'product_name': prod , # random first string from the product_name list
        'category': categ, # take the second string from the product_name pair
        'brand': random.choice(brand),
        'demand': random.choice(demand),
        'price': float(price), # float in order to match cost
        'cost': round(cost,2)
    }
    generated_products.append(product)

products_df = pd.DataFrame(generated_products)
#print(products)
products_df

,sku,product_name,category,brand,demand,price,cost
0,FRU,Lotion,Bodycare,LuxBeauty,Low,18.0,10.80
1,OZV,Sea Salt Spray,Haircare,PamperBeauty,Medium,13.0,7.80
2,XXG,Hair Oil,Haircare,RenewBeauty,Medium,14.0,5.60
3,XWL,Sea Salt Spray,Haircare,RenewBeauty,Medium,6.0,3.90
4,LLI,Bar Soap,Bodycare,LovelyBeauty,Medium,23.0,13.80
...,...,...,...,...,...,...,...
145,SMW,Bond Builder,Haircare,RenewBeauty,Medium,20.0,13.00
146,WVY,Spot Treatment,Skincare,Skinore,Medium,23.0,11.50
147,MIP,Scalp Exfoliator,Haircare,CareForSelf,Low,15.0,8.25
148,PFN,Sunscreen,Skincare,EurBeauty,Low,6.0,2.70


In [135]:
generated_products[:2]

[{'sku': 'EXI',
  'product_name': 'Nail Polish',
  'category': 'Nailcare',
  'brand': 'LuxBeauty',
  'demand': 'Medium',
  'price': 12,
  'cost': 6.6},
 {'sku': 'FEV',
  'product_name': 'Body Scrub',
  'category': 'Bodycare',
  'brand': 'Skinore',
  'demand': 'Low',
  'price': 22,
  'cost': 14.3}]

In [161]:
# ORDER TABLE

# order_id: OR + random 9 unique numbers; generate 10k orders
            # generated_sku_nums = np.random.choice(range(100000,199999), size=10000, replace=False) # replace makes sure we only get unique numbers
            # sku_list = ['ORD' + str(num) for num in generated_sku_nums] THIS IS ORDER LIST!!!

# sku: = product sku BUT if demand=high then take those skus more frequently, if medium then medium, etc.
# quantity: random(1,11), but skew 70% to be < 5.
# date: random dates from 2025-06-01 to 2026-06-01
# unit_product_price: = price

#order_id
generated_order_nums = np.random.choice(range(100000,199999), size=10000, replace=False) # replace makes sure we only get unique numbers
order_list = ['ORD' + str(num) for num in generated_order_nums]

#skus based on demand
high_dem_sku = products_df[products_df['demand'] == 'High']['sku'].tolist() #select sku from products_df where demand='High'
medium_dem_sku = products_df[products_df['demand'] == 'Medium']['sku'].tolist()
low_dem_sku = products_df[products_df['demand'] == 'Low']['sku'].tolist()


orders_final_list = []
for ord in order_list:
    # for each order, we "randomly" select which demand bucket we want. depending on the outcome, we take a random sku from that bucket, and that becomes our item for the order
    random_demand = random.choices(['High', 'Medium', 'Low'], weights=[0.7,0.2,0.1])[0]
    if random_demand == 'High':
        sku = random.choice(high_dem_sku)
    elif random_demand == 'Medium':
        sku = random.choice(medium_dem_sku)
    else:
        sku = random.choice(low_dem_sku)

    random_quant = random.choices(['small', 'big'], weights = [0.7, 0.3])[0]
#    random_quant = random.choices([random.choice(range(1,5)),random.choice(range(5,))], weights = [0.7, 0.3])[0]

    if random_quant == 'small':
        quantity = random.choice(range(1,5))
    else:
        quantity = random.choice(range(5,11))
    
    #unit_price = products_df[products_df['sku']==sku]['price']
    date_range = pd.date_range('2025-06-01', '2026-06-01')

    orders = {
        'order_id': ord,
        'sku': sku,
        'quantity':quantity,
        'date': random.choice(date_range), # CHANGE: only select the date, no hours minutes etc
        'unit_product_price': products_df[products_df['sku']==sku]['price'].iloc[0],
        #'demand_group': random_demand

    }

    orders_final_list.append(orders)


orders_final_list[:10]
# 'sku': random.choices()

# we split the skus from the product table into three lists based on demand and when creating order rows, we assign weights/probabilities based on the demand (0.7,0.2,0.1)


[{'order_id': 'ORD161088',
  'sku': 'ZPH',
  'quantity': 8,
  'date': Timestamp('2025-09-09 00:00:00'),
  'unit_product_price': 10.0},
 {'order_id': 'ORD129966',
  'sku': 'DKY',
  'quantity': 3,
  'date': Timestamp('2025-11-02 00:00:00'),
  'unit_product_price': 14.0},
 {'order_id': 'ORD154275',
  'sku': 'VMX',
  'quantity': 4,
  'date': Timestamp('2026-02-01 00:00:00'),
  'unit_product_price': 20.0},
 {'order_id': 'ORD191276',
  'sku': 'DHT',
  'quantity': 1,
  'date': Timestamp('2026-05-23 00:00:00'),
  'unit_product_price': 17.0},
 {'order_id': 'ORD193670',
  'sku': 'AXT',
  'quantity': 6,
  'date': Timestamp('2026-01-13 00:00:00'),
  'unit_product_price': 20.0},
 {'order_id': 'ORD190872',
  'sku': 'PIK',
  'quantity': 1,
  'date': Timestamp('2025-11-13 00:00:00'),
  'unit_product_price': 12.0},
 {'order_id': 'ORD111785',
  'sku': 'MTU',
  'quantity': 3,
  'date': Timestamp('2025-08-23 00:00:00'),
  'unit_product_price': 11.0},
 {'order_id': 'ORD196890',
  'sku': 'NDW',
  'quantity'

In [160]:
prd = products_df[products_df['sku']==sku]['price']
prd[:10]

133    24.0
Name: price, dtype: float64

In [ ]:
# INVENTORY TABLE

# sku: = product sku
# current_stock: if price < 15 then random(300, 500)
                # else: random (100, 300)
# expiration_date: random from 2026-06-01 to 2026-12-01 BUT 70% of expiration dates should be in the range 2026-06-01 - 2026-10-01


future addition: quantity. some products might be bundles, for instance buying 3 shampoos are cheaper. so shampoo sku: smp, shampoo 3 bundle: smp3.